# VisionGuard — Fine-tune SlowFast R50 for Violence Detection

Fine-tunes a pretrained **SlowFast R50** (Kinetics-400) to binary violence classification.  
The saved model drops directly into VisionGuard via `MODEL_TYPE=slowfast_violence` in `.env`.

## Expected data layout
```
data/
  train/
    violence/        ← MP4 / AVI clips of fights
    nonviolence/     ← MP4 / AVI clips of normal activity
  val/
    violence/
    nonviolence/
```
Recommended dataset: **RWF-2000** (https://github.com/mchengny/RWF2000-Video-Database-for-Violence-Detection)  
– 1000 fight / 1000 non-fight clips from real surveillance footage.

## Output
- `models/slowfast_violence.pt` — full state dict, compatible with VisionGuard
- Set `MODEL_TYPE=slowfast_violence` and `BUFFER_SIZE=32` in `.env` to use it

In [1]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
# Run once; restart kernel after installation.
%pip install pytorchvideo decord torchmetrics scikit-learn matplotlib seaborn --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.transforms import v2 as T

try:
    from decord import VideoReader, cpu as decord_cpu
    _DECORD = True
except ImportError:
    import torchvision.io as tvio
    _DECORD = False
    print("decord not found — falling back to torchvision.io (slower)")

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────────

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path("data")          # change if your data lives elsewhere
TRAIN_DIR   = DATA_ROOT / "train"
VAL_DIR     = DATA_ROOT / "val"
SAVE_PATH   = Path("models/slowfast_violence.pt")
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Classes ───────────────────────────────────────────────────────────────────
CLASSES     = ["nonviolence", "violence"]   # index 0 = non-violent, 1 = violent

# ── SlowFast clip config ──────────────────────────────────────────────────────
# Standard SlowFast R50 config: 32 fast frames, every 4th → 8 slow frames
T_FAST      = 32        # number of frames for the fast pathway
ALPHA       = 4         # slow/fast temporal ratio  (T_SLOW = T_FAST // ALPHA = 8)
FRAME_STEP  = 2         # sample every Nth frame from the raw video (controls speed)
CROP_SIZE   = 224       # spatial crop (SlowFast standard)
RESIZE_TO   = 256       # resize short edge before crop

# ── Training hyperparams ──────────────────────────────────────────────────────
BATCH_SIZE  = 4         # lower if OOM; use GRAD_ACCUM to compensate
GRAD_ACCUM  = 4         # effective batch = BATCH_SIZE × GRAD_ACCUM
NUM_WORKERS = 4
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SEED        = 42

# Phase 1 — train head only (fast, minimal forgetting)
LR_PHASE1   = 1e-3
EPOCHS_P1   = 5

# Phase 2 — unfreeze all layers and fine-tune end-to-end
LR_PHASE2   = 1e-4
EPOCHS_P2   = 15

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Fast frames: {T_FAST}, Slow frames: {T_FAST // ALPHA}")

In [ ]:
# ── 4. Video loading helpers ──────────────────────────────────────────────────

def load_video_frames(path: Path, num_frames: int, frame_step: int) -> np.ndarray:
    """
    Load `num_frames` frames from *path* using a uniform stride of *frame_step*.
    Returns np.ndarray of shape [T, H, W, C] uint8 (RGB).
    """
    need = num_frames * frame_step   # total span of frames required

    if _DECORD:
        vr   = VideoReader(str(path), ctx=decord_cpu(0))
        total = len(vr)
        max_start = max(0, total - need)
        start     = random.randint(0, max_start)
        indices   = list(range(start, min(start + need, total), frame_step))
        # Pad with last frame if clip is shorter than required
        while len(indices) < num_frames:
            indices.append(indices[-1])
        indices = indices[:num_frames]
        frames  = vr.get_batch(indices).asnumpy()          # [T, H, W, C] uint8
    else:
        # torchvision fallback (reads the whole video — slower)
        video, _, _ = tvio.read_video(str(path), pts_unit="sec")
        total = video.shape[0]
        max_start = max(0, total - need)
        start = random.randint(0, max_start)
        indices = list(range(start, min(start + need, total), frame_step))
        while len(indices) < num_frames:
            indices.append(indices[-1])
        indices = indices[:num_frames]
        frames = video[indices].numpy()                    # [T, H, W, C] uint8

    return frames   # [T, H, W, 3]  uint8  RGB


def frames_to_tensor(frames: np.ndarray) -> torch.Tensor:
    """Convert [T, H, W, C] uint8 → [C, T, H, W] float32 in [0, 1]."""
    t = torch.from_numpy(frames).permute(3, 0, 1, 2).float() / 255.0
    return t   # [C, T, H, W]

In [ ]:
# ── 5. Augmentation transforms ────────────────────────────────────────────────
# torchvision v2 transforms operate on [C, T, H, W] or [B, C, T, H, W].

# ImageNet statistics (SlowFast was pretrained on Kinetics which uses these)
_MEAN = [0.45, 0.45, 0.45]
_STD  = [0.225, 0.225, 0.225]

train_transform = T.Compose([
    T.Resize(RESIZE_TO, antialias=True),
    T.RandomCrop(CROP_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1),
    T.Normalize(mean=_MEAN, std=_STD),
])

val_transform = T.Compose([
    T.Resize(RESIZE_TO, antialias=True),
    T.CenterCrop(CROP_SIZE),
    T.Normalize(mean=_MEAN, std=_STD),
])

print("Transforms defined.")

In [ ]:
# ── 6. Dataset ────────────────────────────────────────────────────────────────

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

class ViolenceVideoDataset(Dataset):
    """
    Loads video clips from a class-folder tree and returns
    (slow_pathway, fast_pathway, label) for SlowFast training.

    Folder structure:
        root/
          violence/    ← label 1
          nonviolence/ ← label 0
    """

    def __init__(self, root: Path, transform, t_fast: int = T_FAST,
                 alpha: int = ALPHA, frame_step: int = FRAME_STEP):
        self.transform   = transform
        self.t_fast      = t_fast
        self.alpha       = alpha
        self.frame_step  = frame_step
        self.samples: list = []   # (path, label)

        for label_idx, cls in enumerate(CLASSES):
            cls_dir = root / cls
            if not cls_dir.exists():
                print(f"  WARNING: {cls_dir} not found — skipping class '{cls}'")
                continue
            files = [p for p in cls_dir.rglob("*") if p.suffix.lower() in VIDEO_EXTS]
            self.samples.extend((p, label_idx) for p in files)
            print(f"  {cls:15s}  {len(files):4d} clips  (label={label_idx})")

        print(f"  Total: {len(self.samples)} clips")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            frames = load_video_frames(path, self.t_fast, self.frame_step)
        except Exception as e:
            print(f"[Dataset] Error reading {path}: {e} — using blank clip")
            frames = np.zeros((self.t_fast, CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8)

        clip = frames_to_tensor(frames)           # [C, T_fast, H, W]  float32
        clip = self.transform(clip)               # apply spatial augmentation

        # ── Build SlowFast pathways ─────────────────────────────────────────
        fast = clip                               # [C, T_fast, H, W]
        slow = clip[:, ::self.alpha, :, :]        # [C, T_slow, H, W]

        return slow, fast, label


print("=== Train set ===")
train_ds = ViolenceVideoDataset(TRAIN_DIR, train_transform)
print("=== Val set ===")
val_ds   = ViolenceVideoDataset(VAL_DIR,   val_transform)

In [ ]:
# ── 7. Data loaders (with class-balanced sampler) ─────────────────────────────

def make_weighted_sampler(dataset: ViolenceVideoDataset) -> WeightedRandomSampler:
    """Over-sample the minority class so each epoch sees a balanced mix."""
    labels  = [lbl for _, lbl in dataset.samples]
    counts  = [labels.count(i) for i in range(len(CLASSES))]
    weights = [1.0 / counts[lbl] for lbl in labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=make_weighted_sampler(train_ds),
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")

# Verify shapes
slow_b, fast_b, lbl_b = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  slow: {slow_b.shape}   (B, C, T_slow={T_FAST//ALPHA}, H, W)")
print(f"  fast: {fast_b.shape}   (B, C, T_fast={T_FAST}, H, W)")
print(f"  labels: {lbl_b.tolist()}")

In [ ]:
# ── 8. Visualise sample clips ─────────────────────────────────────────────────

_MEAN_T = torch.tensor([0.45, 0.45, 0.45]).view(3, 1, 1, 1)
_STD_T  = torch.tensor([0.225, 0.225, 0.225]).view(3, 1, 1, 1)

def denorm(x):
    return (x * _STD_T + _MEAN_T).clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle("Fast pathway — 8 evenly-spaced frames per sample", fontsize=12)

for row, (fast, label) in enumerate(zip(fast_b[:2], lbl_b[:2])):
    idxs = np.linspace(0, fast.shape[1] - 1, 8, dtype=int)
    for col, fi in enumerate(idxs):
        img = denorm(fast[:, fi, :, :]).permute(1, 2, 0).numpy()
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(CLASSES[label.item()], fontsize=9, color="red" if label else "green")

plt.tight_layout()
plt.show()

In [ ]:
# ── 9. Model — SlowFast R50 with binary head ──────────────────────────────────
#
# VisionGuard's ClipClassifier passes [B, C, T, H, W] to the model and expects
# [B, 2] softmax output back.  We wrap SlowFast inside SlowFastWrapper which
# splits the input internally — no changes to ClipClassifier needed.

class SlowFastWrapper(nn.Module):
    """
    VisionGuard-compatible wrapper around PyTorchVideo SlowFast R50.

    Input  : [B, C, T_fast, H, W]  float32  (standard VisionGuard clip format)
    Output : [B, 2]  float32  softmax probabilities  [P(nonviolent), P(violent)]

    The slow pathway is derived by taking every `alpha`-th frame from the input.
    """

    def __init__(self, alpha: int = ALPHA, num_classes: int = 2):
        super().__init__()
        self.alpha = alpha

        # Load pretrained SlowFast R50 (Kinetics-400)
        print("Loading SlowFast R50 pretrained on Kinetics-400…")
        self.backbone = torch.hub.load(
            "facebookresearch/pytorchvideo",
            "slowfast_r50",
            pretrained=True,
        )

        # Replace the Kinetics-400 head (400 classes) → binary
        in_features = self.backbone.blocks[-1].proj.in_features
        self.backbone.blocks[-1].proj = nn.Linear(in_features, num_classes)
        nn.init.normal_(self.backbone.blocks[-1].proj.weight, std=0.01)
        nn.init.zeros_(self.backbone.blocks[-1].proj.bias)
        print(f"Head replaced: Linear({in_features}, {num_classes})")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T_fast, H, W]
        slow = x[:, :, ::self.alpha, :, :]   # [B, C, T_slow, H, W]
        logits = self.backbone([slow, x])     # [B, num_classes]
        return torch.softmax(logits, dim=1)   # [B, 2]

    def freeze_backbone(self):
        """Freeze all layers except the final projection head."""
        for name, param in self.backbone.named_parameters():
            param.requires_grad = "blocks.6.proj" in name
        frozen = sum(1 for p in self.backbone.parameters() if not p.requires_grad)
        trainable = sum(1 for p in self.backbone.parameters() if p.requires_grad)
        print(f"Frozen {frozen} params | Trainable {trainable} params")

    def unfreeze_all(self):
        """Unfreeze all layers for full fine-tuning."""
        for param in self.backbone.parameters():
            param.requires_grad = True
        trainable = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        print(f"All layers unfrozen — {trainable/1e6:.1f}M trainable params")


model = SlowFastWrapper(alpha=ALPHA).to(DEVICE)

In [ ]:
# ── 10. Training utilities ────────────────────────────────────────────────────

def accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc, steps = 0.0, 0.0, 0
    all_probs, all_labels = [], []

    for slow, fast, labels in loader:
        fast   = fast.to(DEVICE)
        labels = labels.to(DEVICE)
        probs  = model(fast)                   # [B, 2] — wrapper takes fast only
        loss   = criterion(torch.log(probs + 1e-8), labels)
        total_loss += loss.item()
        total_acc  += accuracy(probs, labels)
        steps += 1
        all_probs.append(probs[:, 1].cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else float("nan")

    return total_loss / steps, total_acc / steps, auc, all_probs, all_labels


def train_one_epoch(model, loader, criterion, optimizer, scaler, grad_accum):
    model.train()
    total_loss, total_acc, steps = 0.0, 0.0, 0
    optimizer.zero_grad()

    for i, (slow, fast, labels) in enumerate(loader):
        fast   = fast.to(DEVICE)
        labels = labels.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            probs = model(fast)                # [B, 2]
            loss  = criterion(torch.log(probs + 1e-8), labels) / grad_accum

        scaler.scale(loss).backward()
        total_loss += loss.item() * grad_accum
        total_acc  += accuracy(probs.detach(), labels)
        steps += 1

        if (i + 1) % grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

    return total_loss / steps, total_acc / steps


criterion = nn.NLLLoss()    # expects log-probabilities from model
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

# History for plots
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_auc": []}

print("Utilities ready.")

In [ ]:
# ── 11. Phase 1 — Train head only ─────────────────────────────────────────────
# Backbone frozen; only the new Linear(in, 2) is updated.
# Learns fast-convergence binary separation without catastrophic forgetting.

model.freeze_backbone()

optimizer_p1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_PHASE1,
)

print(f"\n{'='*60}")
print(f" Phase 1 — Head only  ({EPOCHS_P1} epochs, lr={LR_PHASE1})")
print(f"{'='*60}")

best_val_auc = 0.0

for epoch in range(1, EPOCHS_P1 + 1):
    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, criterion, optimizer_p1, scaler, GRAD_ACCUM
    )
    vl_loss, vl_acc, vl_auc, _, _ = evaluate(model, val_loader, criterion)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)
    history["val_auc"].append(vl_auc)

    print(f"Ep {epoch:2d}/{EPOCHS_P1} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {vl_loss:.4f} acc {vl_acc:.3f} AUC {vl_auc:.3f}")

    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ saved (best AUC {best_val_auc:.3f})")

print("\nPhase 1 complete.")

In [ ]:
# ── 12. Phase 2 — Full fine-tuning ────────────────────────────────────────────
# All layers unlocked; smaller LR to preserve pretrained representations.

model.unfreeze_all()

optimizer_p2 = torch.optim.AdamW(
    model.parameters(), lr=LR_PHASE2, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=EPOCHS_P2, eta_min=1e-6
)

print(f"\n{'='*60}")
print(f" Phase 2 — Full fine-tune  ({EPOCHS_P2} epochs, lr={LR_PHASE2})")
print(f"{'='*60}")

p2_start = len(history["val_auc"])   # offset for history indexing

for epoch in range(1, EPOCHS_P2 + 1):
    tr_loss, tr_acc = train_one_epoch(
        model, train_loader, criterion, optimizer_p2, scaler, GRAD_ACCUM
    )
    vl_loss, vl_acc, vl_auc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)
    history["val_auc"].append(vl_auc)

    lr_now = scheduler.get_last_lr()[0]
    print(f"Ep {epoch:2d}/{EPOCHS_P2} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {vl_loss:.4f} acc {vl_acc:.3f} AUC {vl_auc:.3f} | "
          f"lr {lr_now:.2e}")

    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ saved (best AUC {best_val_auc:.3f})")

print(f"\nPhase 2 complete.  Best val AUC: {best_val_auc:.4f}")

In [ ]:
# ── 13. Training curves ───────────────────────────────────────────────────────

epochs = range(1, len(history["val_auc"]) + 1)
p1_end = EPOCHS_P1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Training history — SlowFast R50 Violence Classifier")

for ax, metric, label in [
    (axes[0], "loss",  "Loss"),
    (axes[1], "acc",   "Accuracy"),
]:
    ax.plot(epochs, history[f"train_{metric}"], label="train", color="steelblue")
    ax.plot(epochs, history[f"val_{metric}"],   label="val",   color="tomato")
    ax.axvline(p1_end, color="grey", ls="--", alpha=0.6, label="Phase 2 start")
    ax.set_xlabel("Epoch")
    ax.set_title(label)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[2].plot(epochs, history["val_auc"], color="darkorange")
axes[2].axvline(p1_end, color="grey", ls="--", alpha=0.6)
axes[2].set_xlabel("Epoch")
axes[2].set_title("Val ROC-AUC")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── 14. Final evaluation — load best checkpoint ───────────────────────────────

model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
_, _, _, all_probs, all_labels = evaluate(model, val_loader, criterion)

threshold = 0.5
preds = (all_probs >= threshold).astype(int)

print(classification_report(all_labels, preds, target_names=CLASSES))

# Confusion matrix
cm = confusion_matrix(all_labels, preds)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title("Confusion Matrix")
axes[0].set_ylabel("True")
axes[0].set_xlabel("Predicted")

# ROC curve
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
auc_score = roc_auc_score(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="darkorange", label=f"AUC = {auc_score:.3f}")
axes[1].plot([0, 1], [0, 1], color="navy", ls="--")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest threshold analysis:")
j_scores = tpr - fpr
best_thresh = thresholds[np.argmax(j_scores)]
print(f"  Youden-optimal threshold: {best_thresh:.3f}")
print(f"  → set CLASSIFIER_THRESHOLD={best_thresh:.2f} in .env")

In [ ]:
# ── 15. Save full model for VisionGuard ───────────────────────────────────────
# Saves both the state dict (for loading) and the wrapper class source
# so VisionGuard can reconstruct the model without this notebook.

torch.save(model.state_dict(), SAVE_PATH)
print(f"State dict saved → {SAVE_PATH}")

# Save training metadata alongside the weights
meta = {
    "classes":          CLASSES,
    "t_fast":           T_FAST,
    "alpha":            ALPHA,
    "frame_step":       FRAME_STEP,
    "crop_size":        CROP_SIZE,
    "best_val_auc":     round(best_val_auc, 4),
    "recommended_threshold": round(float(best_thresh), 3),
    "mean":             _MEAN,
    "std":              _STD,
    "model_type":       "slowfast_violence",
}
meta_path = SAVE_PATH.with_suffix(".json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"Metadata saved   → {meta_path}")

print()
print("═" * 55)
print(" To use this model in VisionGuard, set in .env:")
print(f"   MODEL_TYPE=slowfast_violence")
print(f"   MODEL_PATH=models/slowfast_violence.pt")
print(f"   BUFFER_SIZE={T_FAST}")
print(f"   CLASSIFIER_THRESHOLD={round(float(best_thresh), 2)}")
print("═" * 55)

In [ ]:
# ── 16. Add slowfast_violence to VisionGuard's classifier.py ──────────────────
# Run this cell to patch pipeline/classifier.py so MODEL_TYPE=slowfast_violence works.

CLASSIFIER_PATH = Path("pipeline/classifier.py")
src = CLASSIFIER_PATH.read_text()

INJECTION = '''
        # ── slowfast_violence: fine-tuned binary SlowFast ──────────────────────
        if model_type == "slowfast_violence":
            from train_slowfast import SlowFastWrapper   # defined in notebook
            import config as _cfg
            wrapper = SlowFastWrapper(alpha=4, num_classes=2)
            sd_path = source if isinstance(source, str) and source else _cfg.MODEL_PATH
            if sd_path and Path(sd_path).exists():
                wrapper.load_state_dict(torch.load(sd_path, map_location=self.device))
                print(f"[Classifier] slowfast_violence loaded from {sd_path}")
            return wrapper.to(self.device)
'''

ANCHOR = "        # ── r3d18: binary head, requires fine-tuning to be useful"
if ANCHOR in src and "slowfast_violence" not in src:
    patched = src.replace(ANCHOR, INJECTION + "\n" + ANCHOR)
    CLASSIFIER_PATH.write_text(patched)
    print(f"✓ Patched {CLASSIFIER_PATH} — MODEL_TYPE=slowfast_violence now supported.")
elif "slowfast_violence" in src:
    print("classifier.py already patched.")
else:
    print("Could not find anchor — patch manually by adding the INJECTION block above.")

In [ ]:
# ── 17. Quick inference smoke-test ────────────────────────────────────────────
# Verifies the saved model loads and produces the right output shape.

model_check = SlowFastWrapper(alpha=ALPHA).to(DEVICE)
model_check.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model_check.eval()

dummy = torch.randn(1, 3, T_FAST, CROP_SIZE, CROP_SIZE).to(DEVICE)
with torch.no_grad():
    out = model_check(dummy)

print(f"Input  shape : {dummy.shape}")
print(f"Output shape : {out.shape}   (expected [1, 2])")
print(f"Output probs : {out.cpu().numpy().round(4)}   (should sum to 1.0)")
assert out.shape == (1, 2), "Wrong output shape!"
assert abs(out.sum().item() - 1.0) < 1e-4, "Probabilities don't sum to 1!"
print("\n✓ Model smoke-test passed — ready for VisionGuard.")